# FH-02 · Intent, slots and dialogue state (Phase 2)
Two public datasets, scored **separately** (a gain on one must not hide a loss on the other):
- **MultiWOZ 2.2** → joint goal accuracy (every slot right at a turn) and slot F1; also the subset of turns that rely on history.
- **MASSIVE 1.1** → intent accuracy / macro-F1 and slot F1, English and Hindi.

Caveats to say out loud: MultiWOZ domains (hotel, train, ...) are not calendar tasks - this tests *context handling*, not the product task.
MASSIVE is scored on its own 60-intent labels (no mapping to the project ontology yet).

In [ ]:
# ---- Colab setup: upload futurehuman_testing_suite.zip when asked (once per session) ----
import os, sys, subprocess
SUITE = "/content/futurehuman_testing_suite"
if not os.path.isdir(SUITE):
    from google.colab import files
    up = files.upload()
    subprocess.run(["unzip", "-o", "-q", list(up)[0], "-d", "/content"], check=True)
os.chdir(SUITE); sys.path.insert(0, SUITE)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "accelerate", "bitsandbytes", "tabulate"])
print("ready in", os.getcwd())

In [ ]:
TAG = "base"          # change to "stage1" / "stage2" when you test a trained adapter (same notebook, same tests)
ADAPTER = None        # e.g. "/content/drive/MyDrive/adapters/stage1"  (None = untouched base model)
QUANT = "4bit"        # 4bit | 8bit | fp16
from fh import env
from fh.model import load_qwen, make_gen
env.set_seed(0)
model, tok = load_qwen("Qwen/Qwen3-4B", quant=QUANT, adapter=ADAPTER)
gen = make_gen(model, tok)
info = env.save_snapshot(f"results/env_{TAG}.json", model=model, model_id="Qwen/Qwen3-4B", quant=QUANT, seed=0,
                         extra=dict(adapter=ADAPTER))
info

In [ ]:
QUICK = True                     # True = 60 samples each (~10 min total). False = full sizes (~45 min on T4)
N_MW, N_MS = (60, 60) if QUICK else (150, 200)
import json, pandas as pd
from fh import multiwoz as mw_mod, massive as ms
from fh.report import update_capabilities
from fh.metrics import fmt

## MultiWOZ 2.2

In [ ]:
from datasets import load_dataset
mw = load_dataset("pfb30/multi_woz_v22", trust_remote_code=True)
print(mw)
print(json.dumps(mw["test"][0]["turns"]["frames"][0], indent=1)[:900])   # sanity: confirm the structure the code expects

In [ ]:
slots, intents = mw_mod.build_inventory(mw["train"].select(range(3000)))
print(len(slots), "slot names;", len(intents), "intents"); print(slots[:12])
SYS = mw_mod.system_prompt(slots, intents)
samples = mw_mod.make_samples(mw["test"], n_dialogues=100, max_samples=N_MW, seed=0)
rows = mw_mod.run_multiwoz(samples, gen, SYS)
pd.DataFrame(rows).to_csv(f"results/{TAG}_multiwoz.csv", index=False)
s = mw_mod.summarize(rows)
print("JGA", fmt(s["joint_goal_accuracy"]), "| slot F1", round(s["slot_f1"], 3), "(P %.2f R %.2f)" % (s["slot_precision"], s["slot_recall"]))
print("JGA first turn", fmt(s["jga_first_turn"]), "| JGA with history", fmt(s["jga_with_history"]), "| JSON valid", fmt(s["json_valid"]))

In [ ]:
bad = [r for r in rows if not r["jga"]][:6]
for r in bad: print(r["dialogue_id"], "turn", r["turn"], "| tp/fp/fn", r["tp"], r["fp"], r["fn"], "\n   ", r["raw"][:250])

## MASSIVE 1.1 (English and Hindi)

In [ ]:
slots_en = ms.slot_inventory("en-US")
res, allrows = {}, []
for loc in ["en-US", "hi-IN"]:
    items, names = ms.load_locale(loc, "test", n=N_MS, seed=0)
    SYSM = ms.system_prompt(names, slots_en)
    r_ = ms.run_massive(items, gen, SYSM); allrows += r_; res[loc] = ms.summarize(r_)
    print(loc, "intent acc", fmt(res[loc]["intent_accuracy"]), "| macro-F1", round(res[loc]["intent_macro_f1"], 3), "| slot F1", round(res[loc]["slot_f1"], 3), "| json", fmt(res[loc]["json_valid"]))
pd.DataFrame(allrows).to_csv(f"results/{TAG}_massive.csv", index=False)

In [ ]:
d = pd.DataFrame(allrows); d["ok"] = d.gold_intent == d.pred_intent
print("most confused intents (gold -> pred):"); d[~d.ok].groupby(["locale", "gold_intent", "pred_intent"]).size().sort_values(ascending=False).head(12)

In [ ]:
r = lambda x: dict(value=x["rate"], k=x["k"], n=x["n"])
update_capabilities(TAG, multiwoz_jga=r(s["joint_goal_accuracy"]), multiwoz_slot_f1=dict(value=s["slot_f1"]),
                    massive_intent_acc_en=r(res["en-US"]["intent_accuracy"]), massive_intent_acc_hi=r(res["hi-IN"]["intent_accuracy"]),
                    massive_slot_f1_en=dict(value=res["en-US"]["slot_f1"]), massive_slot_f1_hi=dict(value=res["hi-IN"]["slot_f1"]))